# M3 DeepConvLSTM Slurm Workflow

This notebook is a Slurm-first control surface for Milestone 3 DeepConvLSTM experiments. It submits repo scripts through Slurm, monitors jobs, and inspects generated artifacts with shell tools only.

Guardrails:
- Heavy training, evaluation, dataset loading, quantization, and TFLite export must run through Slurm.
- Do not run user-holdout experiments. E01 remains tooling-only unless explicitly approved later.
- Do not run `pytest` here.
- Do not use `rg`; use `grep`, `find`, `sed`, `tail`, `wc`, `ls`, `squeue`, `sacct`, and `scontrol`.
- The current Arduino merged CSVs are 20 Hz, so E02 is listed but not submitted until a true 100 Hz source is available.


In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
pwd
ls -lh scripts/slurm/submit_m3_matrix_dry_run.sh scripts/slurm/submit_m3_experiment.sh
echo
echo "Working DeepConvLSTM M3 configs:"
for cfg in \
  configs/m3/E00_wisdm_m2_anchor.yaml \
  configs/m3/E03_arduino_downsample_20hz_T100.yaml \
  configs/m3/E04_wisdm_to_g_arduino_g.yaml \
  configs/m3/E05_legacy_arduino_to_mps2.yaml \
  configs/m3/E06_no_norm_matched.yaml \
  configs/m3/E07_skip_inference_norm_diag.yaml \
  configs/m3/E08_T50_window.yaml \
  configs/m3/E09_wisdm_pretrain_arduino_finetune.yaml \
  configs/m3/E10_arduino_from_scratch.yaml
do
  echo "- ${cfg}"
done
echo
echo "Pending true 100 Hz Arduino source, not submitted by default: configs/m3/E02_arduino_zero_shot_100hz_T500.yaml"


## 1. Matrix Dry Run

Set `SUBMIT_DRY_RUN=1` inside the cell before running it to submit config validation through Slurm.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
SUBMIT_DRY_RUN=${SUBMIT_DRY_RUN:-0}
CMD="bash scripts/slurm/submit_m3_matrix_dry_run.sh configs/m3"
echo "$CMD"
if [ "$SUBMIT_DRY_RUN" = "1" ]; then
  eval "$CMD"
else
  echo "Set SUBMIT_DRY_RUN=1 in this cell to submit."
fi


## 2. QAT Smoke Runs

These prove FP32 TFLite export/eval, PTQ INT8, QAT INT8, dataset metadata, transfer normalization, and M3 reporting. Set `SUBMIT_SMOKES=1` inside the cell to submit them.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
SUBMIT_SMOKES=${SUBMIT_SMOKES:-0}
SMOKE_ARGS="--smoke --max-windows-per-class 20 --representative-samples 16 --timing-warmup-samples 2 --timing-timed-samples 8"
CONFIGS=(
  configs/m3/E00_wisdm_m2_anchor.yaml
  configs/m3/E03_arduino_downsample_20hz_T100.yaml
  configs/m3/E04_wisdm_to_g_arduino_g.yaml
  configs/m3/E05_legacy_arduino_to_mps2.yaml
  configs/m3/E06_no_norm_matched.yaml
  configs/m3/E07_skip_inference_norm_diag.yaml
  configs/m3/E08_T50_window.yaml
  configs/m3/E09_wisdm_pretrain_arduino_finetune.yaml
  configs/m3/E10_arduino_from_scratch.yaml
)
SUFFIXES=(smoke_qat_e00 smoke_qat_e03 smoke_qat_e04 smoke_qat_e05 smoke_qat_e06 smoke_qat_e07 smoke_qat_e08 smoke_qat_e09 smoke_qat_e10)
for i in "${!CONFIGS[@]}"; do
  CMD="bash scripts/slurm/submit_m3_experiment.sh ${CONFIGS[$i]} ${SMOKE_ARGS} --artifact-suffix ${SUFFIXES[$i]}"
  echo "$CMD"
  if [ "$SUBMIT_SMOKES" = "1" ]; then eval "$CMD"; fi
done
if [ "$SUBMIT_SMOKES" != "1" ]; then echo "Set SUBMIT_SMOKES=1 in this cell to submit."; fi


## 3. Full DeepConvLSTM Runs

Set `SUBMIT_FULL_RUN=1` inside the cell to submit the full run set. E01 user-holdout and E02 true-100-Hz are intentionally excluded.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
SUBMIT_FULL_RUN=${SUBMIT_FULL_RUN:-0}
# Optional: set M3_SLURM_EXCLUDE before running this cell if Slurm assigns unhealthy GPU nodes.
CONFIGS=(
  configs/m3/E00_wisdm_m2_anchor.yaml
  configs/m3/E03_arduino_downsample_20hz_T100.yaml
  configs/m3/E04_wisdm_to_g_arduino_g.yaml
  configs/m3/E05_legacy_arduino_to_mps2.yaml
  configs/m3/E06_no_norm_matched.yaml
  configs/m3/E07_skip_inference_norm_diag.yaml
  configs/m3/E08_T50_window.yaml
  configs/m3/E09_wisdm_pretrain_arduino_finetune.yaml
  configs/m3/E10_arduino_from_scratch.yaml
)
SUFFIXES=(full_e00 full_e03 full_e04 full_e05 full_e06 full_e07 full_e08 full_e09 full_e10)
for i in "${!CONFIGS[@]}"; do
  CMD="bash scripts/slurm/submit_m3_experiment.sh ${CONFIGS[$i]} --artifact-suffix ${SUFFIXES[$i]}"
  echo "$CMD"
  if [ "$SUBMIT_FULL_RUN" = "1" ]; then eval "$CMD"; fi
done
if [ "$SUBMIT_FULL_RUN" != "1" ]; then echo "Set SUBMIT_FULL_RUN=1 in this cell to submit."; fi


## 4. Monitor Jobs

Use `JOBIDS="7186 7187"` in the cell when you want accounting and logs for specific runs.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
squeue -u $USER
JOBIDS=${JOBIDS:-}
if [ -n "$JOBIDS" ]; then
  IDS=$(echo "$JOBIDS" | sed 's/[[:space:]][[:space:]]*/,/g')
  sacct -j "$IDS" --format=JobID,JobName%45,State,ExitCode,Elapsed,Timelimit,Start,End -P
  for job in $JOBIDS; do
    echo "===== logs for ${job} ====="
    find slurm_logs -type f | grep "\\.${job}\\." || true
  done
else
  echo "Set JOBIDS='7186 7187 ...' in this cell to inspect specific jobs."
fi


In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
JOBIDS=${JOBIDS:-}
if [ -z "$JOBIDS" ]; then echo "Set JOBIDS='7186 7187 ...' in this cell to tail logs."; exit 0; fi
for job in $JOBIDS; do
  find slurm_logs -type f | grep "\\.${job}\\." | while read -r f; do
    echo "===== $f ====="
    tail -n 80 "$f"
  done
done


## 5. Artifact Checks

This inspects generated M3 reports, datacards, and TFLite exports without loading raw datasets on the host.

In [ ]:
%%bash
set -euo pipefail
cd /shared/b00088568/github/har-mcu
for suffix in full_e00 full_e03 full_e04 full_e05 full_e06 full_e07 full_e08 full_e09 full_e10 smoke_qat_e00 smoke_qat_e03 smoke_qat_e09 smoke_qat_e10; do
  report="reports/m3/${suffix}/m3_experiment_master.csv"
  if [ -f "$report" ]; then
    echo "===== ${report} ====="
    sed -n '1,3p' "$report"
  fi
done
echo "===== TFLite exports ====="
find models_tflite -type f | grep -E 'm3|full_e|smoke_qat' | grep '\\.tflite$' | tail -n 80 || true
echo "===== Datacards ====="
find data/processed reports/m3 -type f | grep 'datacard_.*\\.json$' | grep -E 'full_e|smoke_qat' | tail -n 80 || true
